<a href="https://colab.research.google.com/github/jessgab-prog/FUNDAI-Laboratories-Getaruelas/blob/main/Lab_5_Design_a_Rule_Based_Expert_System_Getaruelas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Design a Rule-Based Expert System

## Fundamentals of Artificial Intelligence

**Name:** Getaruelas, Jess Gabriel A.
**Course:** BSIT  
**Section:** 09282-FUNDAI  
**Date:** 9/25/2026  

**Domain:** Medical Symptom Triage Bot  
**GitHub URL:** https://github.com/jessgab-prog/FUNDAI-Laboratories-Getaruelas

## Disclaimer  
This expert system is for educational purposes only.  
It is not a real medical diagnostic system.

## Knowledge Acquisition

### Domain
Medical symptom triage[cite: 16, 18].

### Source of Knowledge
- General common-sense health rules[cite: 16, 18]
- Instructor-approved sample rules[cite: 16, 18]
- Basic first-aid recommendations[cite: 16, 18]
- Educational examples only[cite: 16, 18]

### Rules Collected
1. Fever, cough, and body ache may suggest flu-like symptoms[cite: 16, 18].
2. Cough and shortness of breath require medical consultation[cite: 16, 18].
3. Severe chest pain requires emergency care[cite: 16, 18].
4. Suspected flu recommends rest and hydration[cite: 16, 18].

### Limitations
- This system is not a real medical diagnostic tool[cite: 16, 18].
- The rules are simplified[cite: 16, 18].
- The system does not consider medical history, age, allergies, or severity[cite: 16, 18].
- A real medical expert system requires validation by licensed professionals[cite: 16, 18].

In [13]:
RULES = [
    {
        "id": "R1",
        "priority": 10,
        "if": {
            "fever": True,
            "cough": True,
            "body_ache": True
        },
        "then": {
            "flu_like": True
        },
        "description": "Fever, cough, and body ache suggest flu-like symptoms."
    },
    {
        "id": "R2",
        "priority": 9,
        "if": {
            "fever": True,
            "sore_throat": True
        },
        "then": {
            "throat_infection_possible": True
        },
        "description": "Fever and sore throat suggest possible throat infection."
    },

    {
        "id": "R3",
        "priority": 9,
        "if": {
            "cough": True,
            "shortness_of_breath": True
        },
        "then": {
            "respiratory_warning": True
        },
        "description": "Cough and shortness of breath require medical attention."
    },
    {
        "id": "R4",
        "priority": 8,
        "if": {
            "flu_like": True
        },
        "then": {
            "suspect_flu": True
        },
        "description": "Flu-like symptoms lead to suspected flu."
    },
    {
        "id": "R5",
        "priority": 7,
        "if": {
            "suspect_flu": True
        },
        "then": {
            "recommend_rest_and_hydration": True
        },
        "description": "Suspected flu recommends rest and hydration."
    },

    {
        "id": "R6",
        "priority": 10,
        "if": {
            "respiratory_warning": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Respiratory warning recommends doctor consultation."
    },
    {
        "id": "R7",
        "priority": 12,
        "if": {
            "severe_chest_pain": True
        },
        "then": {
            "emergency": True
        },
        "description": "Severe chest pain requires emergency care."
    },
    {
        "id": "R8",
        "priority": 6,
        "if": {
            "throat_infection_possible": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Possible throat infection recommends doctor consultation."
    }
]

In [14]:
DEFAULT_FACTS = {
    "fever": False,
    "cough": False,
    "body_ache": False,
    "sore_throat": False,
    "shortness_of_breath": False,
    "severe_chest_pain": False
}

In [15]:
def make_facts(**overrides):
    facts = dict(DEFAULT_FACTS)
    facts.update(overrides)
    return facts

In [16]:
def rule_condition_matches(rule_if, facts):
    for fact_name, required_value in rule_if.items():
        if facts.get(fact_name, False) != required_value:
            return False
    return True

In [17]:
def forward_chain(facts, rules, trace=True):
    facts = dict(facts)
    fired_rules = []

    sorted_rules = sorted(rules, key=lambda rule: rule.get("priority", 0), reverse=True)

    changed = True

    while changed:
        changed = False

        for rule in sorted_rules:
            if rule["id"] in fired_rules:
                continue

            if rule_condition_matches(rule["if"], facts):
                if trace:
                    print(f"Rule {rule['id']} fired.")
                    print(f"Description: {rule.get('description', 'No description')}")
                    print(f"New facts added: {rule['then']}")
                    print("-" * 50)

                for fact_name, fact_value in rule["then"].items():
                    if facts.get(fact_name) != fact_value:
                        facts[fact_name] = fact_value
                        changed = True

                fired_rules.append(rule["id"])

    return facts, fired_rules

In [18]:
def explain_inference(fired_rule_ids, rules):
    rule_map = {rule["id"]: rule for rule in rules}

    print("=" * 60)
    print("EXPLANATION OF INFERENCE")
    print("=" * 60)

    if len(fired_rule_ids) == 0:
        print("No rules were fired.")
        return

    for rule_id in fired_rule_ids:
        rule = rule_map[rule_id]

        print(f"Rule {rule_id}")
        print(f"IF: {rule['if']}")
        print(f"THEN: {rule['then']}")
        print(f"Reason: {rule.get('description', 'No description')}")
        print("-" * 60)

In [19]:
ADVICE_PRIORITY = [
    (
        "emergency",
        "EMERGENCY: Please seek immediate medical help."
    ),
    (
        "recommend_doctor",
        "RECOMMENDATION: Please consult a doctor."
    ),
    (
        "suspect_flu",
        "POSSIBLE CONDITION: Flu-like illness."
    ),
    (
        "throat_infection_possible",
        "POSSIBLE CONDITION: Possible throat infection."
    ),
    (
        "respiratory_warning",
        "WARNING: Respiratory symptoms detected."
    ),
    (
        "recommend_rest_and_hydration",
        "ADVICE: Rest and stay hydrated."
    )
]

def get_conclusions(facts):
    conclusions = []

    for fact_name, message in ADVICE_PRIORITY:
        if facts.get(fact_name, False):
            conclusions.append(message)

    if len(conclusions) == 0:
        conclusions.append(
            "No specific condition matched. "
            "Please provide more symptoms or consult a professional."
        )

    return conclusions

In [20]:
def run_case(case_name, facts):
    print("=" * 60)
    print(f"CASE: {case_name}")
    print("=" * 60)
    print("Input facts:")
    print(facts)
    print("-" * 60)

    final_facts, fired_rules = forward_chain(facts, RULES, trace=True)

    print("Final facts:")
    print(final_facts)

    print("\nConclusions:")
    for conclusion in get_conclusions(final_facts):
        print("-", conclusion)

    print()

    explain_inference(fired_rules, RULES)

    print("\n\n")

In [21]:
# Case 1: Flu-like symptoms
case1 = make_facts(
    fever=True,
    cough=True,
    body_ache=True
)

# Case 2: Respiratory warning
case2 = make_facts(
    cough=True,
    shortness_of_breath=True
)

# Case 3: Emergency condition
case3 = make_facts(
    severe_chest_pain=True
)

# Case 4: No matching symptoms
case4 = make_facts()

run_case("Flu-like symptoms", case1)
run_case("Respiratory warning", case2)
run_case("Emergency chest pain", case3)
run_case("No symptoms", case4)

CASE: Flu-like symptoms
Input facts:
{'fever': True, 'cough': True, 'body_ache': True, 'sore_throat': False, 'shortness_of_breath': False, 'severe_chest_pain': False}
------------------------------------------------------------
Rule R1 fired.
Description: Fever, cough, and body ache suggest flu-like symptoms.
New facts added: {'flu_like': True}
--------------------------------------------------
Rule R4 fired.
Description: Flu-like symptoms lead to suspected flu.
New facts added: {'suspect_flu': True}
--------------------------------------------------
Rule R5 fired.
Description: Suspected flu recommends rest and hydration.
New facts added: {'recommend_rest_and_hydration': True}
--------------------------------------------------
Final facts:
{'fever': True, 'cough': True, 'body_ache': True, 'sore_throat': False, 'shortness_of_breath': False, 'severe_chest_pain': False, 'flu_like': True, 'suspect_flu': True, 'recommend_rest_and_hydration': True}

Conclusions:
- POSSIBLE CONDITION: Flu-lik

In [22]:
def ask_yes_no(question):
    answer = input(question + " (yes/no): ")
    answer = answer.strip().lower()
    return answer in ["yes", "y", "true", "1"]


def collect_facts():
    facts = dict(DEFAULT_FACTS)

    print("Medical Symptom Triage Bot")
    print("Answer the following questions.")
    print("-" * 40)

    facts["fever"] = ask_yes_no("Do you have fever?")
    facts["cough"] = ask_yes_no("Do you have cough?")
    facts["body_ache"] = ask_yes_no("Do you have body ache?")
    facts["sore_throat"] = ask_yes_no("Do you have sore throat?")
    facts["shortness_of_breath"] = ask_yes_no("Do you have shortness of breath?")
    facts["severe_chest_pain"] = ask_yes_no("Do you have severe chest pain?")

    return facts


def interactive_mode():
    facts = collect_facts()

    final_facts, fired_rules = forward_chain(facts, RULES, trace=True)

    print("\nConclusions:")
    for conclusion in get_conclusions(final_facts):
        print("-", conclusion)

    explain_inference(fired_rules, RULES)

In [ ]:
interactive_mode()

Medical Symptom Triage Bot
Answer the following questions.
----------------------------------------


## Guide Questions and Answers

### 1. What is an expert system?
**Answer:** An expert system is a computer program that emulates the decision-making ability of a human expert in a specific domain. It uses a knowledge base of rules and facts along with an inference engine to reason through data and reach logical conclusions.

### 2. What are the main components of an expert system?
**Answer:** The main components include:
- **User Interface:** Allows the user to interact with the system and input data.
- **Knowledge Base:** Stores domain-specific facts and IF-THEN rules.
- **Inference Engine:** Applies logical rules to facts to deduce new information.
- **Explanation Facility:** Clarifies how and why specific conclusions were reached.

### 3. What is forward chaining?
**Answer:** Forward chaining is a data-driven reasoning strategy that starts with known facts and applies IF-THEN rules iteratively to extract or infer new facts until no further conclusions can be drawn.

### 4. How does your system use forward chaining?
**Answer:** Our system uses forward chaining via the `forward_chain()` function. It accepts a set of initial symptom facts and iterates through priority-sorted rules. When rule conditions match current facts, its `then` block adds new derived facts (e.g., `flu_like` or `respiratory_warning`) to the state, repeating until no new facts are generated.

### 5. Which rules fired in Test Case 1?
**Answer:** Rules **R1** (`fever` + `cough` + `body_ache` → `flu_like`), **R4** (`flu_like` → `suspect_flu`), and **R5** (`suspect_flu` → `recommend_rest_and_hydration`) fired sequentially.

### 6. Which rules fired in Test Case 2?
**Answer:** Rules **R3** (`cough` + `shortness_of_breath` → `respiratory_warning`) and **R6** (`respiratory_warning` → `recommend_doctor`) fired sequentially.

### 7. Why is an explanation facility important?
**Answer:** An explanation facility provides transparency and builds trust by detailing the logical rule chain behind each output. This prevents the system from operating as an opaque "black box" and allows users or experts to verify correctness.

### 8. What challenges did you encounter while designing the rules?
**Answer:** Key challenges included assigning rule priority levels so severe conditions evaluate first, managing multi-step rule dependencies without causing logic loops, and ensuring clean default states for boolean symptom facts.

### 9. What are the limitations of your expert system?
**Answer:** It uses binary boolean facts without handling uncertainty or probabilities, relies on a simplified set of only 8 rules, cannot learn dynamically, and ignores medical history, symptom duration, or patient age.

### 10. Why should this system not be used for real medical diagnosis?
**Answer:** It is an educational prototype built with simplified rules. Real medical triage requires licensed healthcare professionals, comprehensive clinical validation, physical examinations, and diagnostic testing.

## Reflection
- Building this Medical Symptom Triage Bot provided direct practical experience with symbolic AI and forward-chaining inference engines. Writing the engine from scratch clarified how rule priorities and logical condition-matching resolve conflicts and derive dynamic, multi-step inferences. Developing the explanation facility underscored the critical role of explainability and transparency in AI, especially within healthcare-adjacent systems. Overall, this activity effectively demonstrated the core strengths and inherent boundaries of rule-based systems.

## Conclusion
- In this laboratory activity, a custom rule-based expert system was successfully developed using Python. The system effectively processes patient symptom inputs through a forward-chaining inference engine, evaluates rules by priority, derives tiered recommendations, and provides detailed inference explanations. All test cases executed correctly, confirming the reliability of symbolic reasoning in structured decision-making domains while highlighting the absolute necessity of safety disclaimers.